In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

# ==========================================================
# 1. DATA GENERATION
# ==========================================================
np.random.seed(42)
n_samples = 1000

data = {
    'student_id': range(1, n_samples + 1),
    'department': np.random.choice(['CSE', 'IT', 'ECE', 'EEE', 'MECH'], n_samples),
    'cgpa': np.clip(np.random.normal(7.5, 1.2, n_samples), 5.0, 10.0),
    'active_backlogs': np.random.randint(0, 5, n_samples),
    'skill_stack': np.random.choice(['Web Dev', 'Data Science', 'Cloud', 'Core', 'Cyber'], n_samples),
    'internships': np.random.randint(0, 4, n_samples),
    'projects': np.random.randint(0, 10, n_samples),
    'aptitude_score': np.random.randint(40, 100, n_samples),
    'communication_score': np.random.randint(1, 11, n_samples)
}

df = pd.DataFrame(data)

# Placement Logic (Hidden Rule)
def determine_placement(row):
    score = (row['cgpa'] * 10) + (row['internships'] * 15) + (row['projects'] * 2) + \
            (row['aptitude_score'] * 0.3) + (row['communication_score'] * 2) - \
            (row['active_backlogs'] * 15)
    if row['department'] in ['CSE', 'IT']:
        score += 5
    return 1 if score > 75 else 0

df['placed'] = df.apply(determine_placement, axis=1)

# ==========================================================
# 2. STRONG FEATURE ENGINEERING
# ==========================================================

df['composite_score'] = (
    df['cgpa'] * 10 +
    df['internships'] * 15 +
    df['projects'] * 2 +
    df['aptitude_score'] * 0.3 +
    df['communication_score'] * 2 -
    df['active_backlogs'] * 18
)

df['cgpa_x_internship'] = df['cgpa'] * df['internships']
df['aptitude_x_projects'] = df['aptitude_score'] * df['projects']

# ==========================================================
# 3. PREPARATION
# ==========================================================

X = df.drop(['student_id', 'placed'], axis=1)
y = df['placed']

categorical_cols = ['department', 'skill_stack']
numerical_cols = [
    'cgpa', 'internships', 'projects', 'aptitude_score',
    'communication_score', 'active_backlogs',
    'composite_score', 'cgpa_x_internship',
    'aptitude_x_projects'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# ==========================================================
# 4. SPLIT + PREPROCESS + SMOTE (MODIFIED)
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Apply preprocessing BEFORE SMOTE
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_processed, y_train)

# ==========================================================
# 5. OPTIMIZED XGBOOST MODEL (MODIFIED)
# ==========================================================

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    min_child_weight=3,
    reg_alpha=0.5,
    reg_lambda=1,
    random_state=42,
    use_label_encoder=False
)

# Since preprocessing is done separately, the pipeline only needs the classifier
pipeline = Pipeline(steps=[
    ('classifier', model)
])

# ==========================================================
# 6. TRAIN (MODIFIED)
# ==========================================================

pipeline.fit(X_train_res, y_train_res)

# ==========================================================
# 7. PREDICT (MODIFIED)
# ==========================================================
import joblib
joblib.dump(pipeline, "xgboost_placement_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")
print("✅ Model and Preprocessor saved successfully!")
y_pred = pipeline.predict(X_test_processed)
accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print(f"🏆 FINAL ACCURACY: {accuracy*100:.2f}%")
print("="*60)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))
# ... (Keep all your existing training code above) ...

# ==========================================================
# 8. SAVE MODEL & PREPROCESSOR (NEW UPDATE)
# ==========================================================
import joblib

# We save the full pipeline (which includes the classifier)
joblib.dump(pipeline, "xgboost_placement_model.pkl")

# We save the preprocessor separately so the API can scale new input data
joblib.dump(preprocessor, "preprocessor.pkl")

print("\n" + "="*60)
print("✅ SUCCESS: Model and Preprocessor saved successfully!")
print("Files created: xgboost_placement_model.pkl, preprocessor.pkl")
print("="*60)


c:\Users\sweth\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [16:53:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Model and Preprocessor saved successfully!

🏆 FINAL ACCURACY: 98.00%

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.92      0.92        25
           1       0.99      0.99      0.99       175

    accuracy                           0.98       200
   macro avg       0.95      0.95      0.95       200
weighted avg       0.98      0.98      0.98       200


✅ SUCCESS: Model and Preprocessor saved successfully!
Files created: xgboost_placement_model.pkl, preprocessor.pkl
